In [1]:
import torch
from torch import nn
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from model.multi_headed_attention import MultiHeadedAttention
from model.vanilla_neural_network import VanillaNeuralNetwork

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
class Encoder_Block(nn.Module):
    def __init__(self, model_dim: int, num_heads: int):
        super().__init__()
        self.mhsa = MultiHeadedAttention(model_dim, num_heads, mask=False)  # Mask is False for encoder
        self.vanilla_nn = VanillaNeuralNetwork(model_dim)
        self.layer_norm_one = nn.LayerNorm(model_dim)
        self.layer_norm_two = nn.LayerNorm(model_dim)

    def forward(self, embedded):
        embedded = embedded + self.mhsa(self.layer_norm_one(embedded)) # skip connection
        embedded = embedded + self.vanilla_nn(self.layer_norm_two(embedded)) # another skip connection
        return embedded

In [ ]:
model_dim = 4 # The dimension for embeddings and attention, the same number is often used for both, model_dim > 0
num_heads = 2 # The number of self-attention instances, num_head > 0, and model_dim % num_heads = 0. The input and output shapes do not depend on num_heads
encoder_block = Encoder_Block(model_dim, num_heads)
encoder_block = encoder_block.to(device)
embedded = [
    [[-0.6775, 1.4919, 0.8760, 0.9440],
    [0.4388, 0.5290, -0.2510, -1.2941]],
    [[2.0576, 0.6107, -0.7395, -0.2010],
    [0.4728, 1.0233, -0.9400, 2.0409]]
] # BxTxA
embedded = torch.tensor(embedded, dtype=torch.float32)
embedded = embedded.to(device)

output = encoder_block(embedded)
print(output)

tensor([[[-0.6508,  1.5484,  0.8760,  1.0251],
         [ 0.6365,  0.3074, -0.5600, -1.5848]],

        [[ 2.5371,  0.7574, -1.2624, -0.7763],
         [ 0.8486,  1.2173, -1.6011,  1.3452]]], device='cuda:0',
       grad_fn=<AddBackward0>)
